In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import json
import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 9})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, load_excitation_result, QUANTITATIVE_RESULTS_ROOT
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result,
    evaluate_model_prediction, evaluate_model_rollout, evaluate_persistence_prediction
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

In [ ]:
exp_name = "submission_nov"

filter_testing_points = True
tag = "safely_excitable" if filter_testing_points else "full_box"

show_baseline = False

name = f"modeling_results_{exp_name}_{tag}"

# TODO:

- show rollout performance comparison
- show one-step performance comparsion

In [ ]:
# if (QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet").exists():
#     modeling_results = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet")
# else:
#     raise

# modeling_results = modeling_results.loc[
#     modeling_results["excitation_objective"].ne("dmpe_static_LOLA_ls")
# ].copy()

USE_CURV64 = True

curvature_objectives = [
    "dmpe_static_curv",
    "dmpe_static_mixture_curv",
]

modeling_results = pd.read_parquet(
    QUANTITATIVE_RESULTS_ROOT / f"{name}.parquet"
)

def is_replacement(df):
    return (
        df["system_name"].eq("pmsm_nominal")
        & df["model_type"].eq("gru")
        & df["excitation_objective"].isin(curvature_objectives)
    )

if USE_CURV64:
    new_name = (
        f"modeling_results_submission_nov_curv64_{tag}.parquet"
    )
    new = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / new_name)
    new = new.loc[is_replacement(new)].copy()

    counts = (
        new.groupby("excitation_objective").size()
        .reindex(curvature_objectives, fill_value=0)
    )
    if not counts.eq(15).all():
        raise ValueError(
            f"Expected 15 GRUs per curvature objective; "
            f"found {counts.to_dict()}"
        )

    modeling_results = pd.concat(
        [
            modeling_results.loc[~is_replacement(modeling_results)],
            new,
        ],
        ignore_index=True,
    )

modeling_results = modeling_results.loc[
    modeling_results["excitation_objective"].ne("dmpe_static_LOLA_ls")
].copy()

In [ ]:
from admpe.evaluation.modeling_visualization import visualize_model_prediction_metric_boxplots, visualize_model_rollout_metric_boxplots

In [ ]:
modeling_results

In [ ]:
def plot_rollout_training_horizon(df, signal="aprbs"):
    systems = ("fluid_tank", "pendulum", "cart_pole", "pmsm_nominal")
    titles = ("Fluid tank", "Pendulum", "Cart-pole", "PMSM")
    objective_labels = {
        "dmpe_uniform": "Uniform",
        "dmpe_static_dynamics": "Dyn.",
        "dmpe_static_mixture_dynamics": "Mix dyn.",
        "dmpe_static_curv": "Curv.",
        "dmpe_static_mixture_curv": "Mix curv.",
        # "dmpe_static_LOLA_ls": "LOLA",

    }
    column = f"rollout.{signal}.metrics.rmse_by_step"
    if column not in df:
        raise ValueError(f"No saved rollout signal {signal!r}.")

    selected = df.loc[df["system_name"].isin(systems)].copy()
    selected["horizon"] = [
        int(json.loads(
            (pathlib.Path(path) / "experiment.json").read_text()
        )["params"]["model_trainer_params"]["sequence_length"]) - 1
        for path in selected["path"]
    ]
    horizons = selected.groupby("system_name")["horizon"].agg(["min", "max"])
    if (horizons["min"] != horizons["max"]).any():
        raise ValueError("The selected runs have different training horizons within a system.")

    records = []
    for _, run in selected.iterrows():
        values = run[column]
        if values is None or (np.ndim(values) == 0 and pd.isna(values)):
            continue
        values = np.asarray(values, dtype=float)
        h = int(run["horizon"])
        if values.ndim != 1 or not 1 <= h <= values.size:
            raise ValueError(f"{run['exp_id']}: cannot evaluate horizon {h}.")
        mse = float(np.mean(values[:h] ** 2))
        if not np.isfinite(mse) or mse <= 0:
            raise ValueError(f"{run['exp_id']}: log-scale MSE must be finite and positive.")
        records.append({
            "system_name": run["system_name"],
            "excitation_objective": run["excitation_objective"],
            "exp_id": run["exp_id"], "horizon": h, "mse": mse,
        })
    metrics = pd.DataFrame(records, columns=[
        "system_name", "excitation_objective", "exp_id", "horizon", "mse",
    ])

    fig, axes = plt.subplots(
        1, 4, figsize=(figurewidth_in, 2), layout="constrained",
    )
    for ax, system, title in zip(axes, systems, titles):
        h = int(horizons.loc[system, "min"])
        ax.set_title(f"{title}\n$h={h}$", fontsize=9)
        system_metrics = metrics.loc[metrics["system_name"].eq(system)]
        if system_metrics.empty:
            ax.text(0.5, 0.5, f"No {signal.upper()} data", transform=ax.transAxes,
                    ha="center", va="center", fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        for position, objective in enumerate(objective_labels):
            values = system_metrics.loc[
                system_metrics["excitation_objective"].eq(objective), "mse",
            ].to_numpy()
            if values.size:
                color = plt.get_cmap("tab10")(position)
                ax.boxplot(
                    [values], positions=[position], widths=0.6,
                    patch_artist=True, manage_ticks=False,
                    boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.6},
                    medianprops={"color": "black"},
                    flierprops={"marker": ".", "markeredgecolor": color},
                    showfliers=False
                )
        ax.set_xticks(range(len(objective_labels)), list(objective_labels.values()),
                      rotation=65, ha="right", fontsize=8)
        ax.set_xlim(-0.6, len(objective_labels) - 0.4)
        ax.set_yscale("log")
        if system_metrics["mse"].max() / system_metrics["mse"].min() < 10:
            ax.yaxis.set_major_locator(
                mpl.ticker.LogLocator(base=10, subs=(1, 2, 5), numticks=10)
            )
            ax.yaxis.set_major_formatter(mpl.ticker.LogFormatterSciNotation(
                labelOnlyBase=False, minor_thresholds=(np.inf, np.inf),
            ))
        else:
            ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10, numticks=4))
        ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
        ax.tick_params(axis="y", labelsize=8)
        ax.grid(axis="y", which="major", alpha=0.3)
        ax.set_axisbelow(True)
    fig.supylabel("Rollout MSE", fontsize=9)
    fig.suptitle(f"{signal.upper()} rollout at the training horizon", fontsize=9)
    return fig, axes, metrics

In [ ]:
for SIGNAL in ["aprbs", "multisine", "ocp", "pi"]:
    fig, axes, rollout_metrics_at_training_horizon = plot_rollout_training_horizon(
        modeling_results, signal=SIGNAL,
    )

# 4x4:

In [ ]:
import json
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def plot_rollout_training_horizon(df, signal=None, plot_style="boxplot", jitter=0.2, *, horizon=None):
    """Plot cached training-horizon MSE: signal rows by system columns.

    signal=None selects all four signals; a string selects one row.
    A sequence of strings selects a custom set of rows.
    plot_style is "boxplot" or "scatter"; jitter controls horizontal point spread.
    """
    if plot_style not in {"boxplot", "scatter"}:
        raise ValueError("plot_style must be 'boxplot' or 'scatter'.")
    if not 0 <= jitter < 0.5:
        raise ValueError("jitter must be at least 0 and less than 0.5.")
    rng = np.random.default_rng(0)
    systems = ("fluid_tank", "pendulum", "cart_pole", "pmsm_nominal")
    titles = ("Fluid tank", "Pendulum", "Cart-pole", "PMSM")
    signals = (
        ("aprbs", "multisine", "ocp", "pi") if signal is None
        else (signal,) if isinstance(signal, str)
        else tuple(signal)
    )
    if not signals or len(set(signals)) != len(signals):
        raise ValueError("Select at least one signal, without duplicates.")
    columns = {s: f"rollout.{s}.metrics.rmse_by_step" for s in signals}
    missing = [s for s, column in columns.items() if column not in df]
    if missing:
        raise ValueError(f"No saved rollout signals: {missing}")

    objective_labels = {
        "dmpe_uniform": "Uniform",
        "dmpe_static_dynamics": "Dynamics",
        "dmpe_static_mixture_dynamics": "Mixture dynamics",
        "dmpe_static_curv": "Curvature",
        "dmpe_static_mixture_curv": "Mixture curvature",
        # "dmpe_static_LOLA_ls": "LOLA",
    }
    selected = df.loc[df["system_name"].isin(systems)].copy()
    if horizon is None:
        selected["horizon"] = [
            int(json.loads(
                (pathlib.Path(path) / "experiment.json").read_text()
            )["params"]["model_trainer_params"]["sequence_length"]) - 1
            for path in selected["path"]
        ]
    else:
        if (
            isinstance(horizon, (bool, np.bool_))
            or not isinstance(horizon, (int, np.integer))
            or horizon < 1
        ):
            raise ValueError("horizon must be None or a positive integer.")
        selected["horizon"] = int(horizon)
    horizons = selected.groupby("system_name")["horizon"].agg(["min", "max"])
    if set(systems).difference(horizons.index):
        raise ValueError("The dataframe must contain modeling runs for all four systems.")
    if (horizons["min"] != horizons["max"]).any():
        raise ValueError("The selected runs have different training horizons within a system.")

    records = []
    for _, run in selected.iterrows():
        h = int(run["horizon"])
        for s, column in columns.items():
            values = run[column]
            if values is None or (np.ndim(values) == 0 and pd.isna(values)):
                continue
            values = np.asarray(values, dtype=float)
            if values.ndim != 1 or not 1 <= h <= values.size:
                raise ValueError(f"{run['exp_id']}: cannot evaluate horizon {h}.")
            mse = float(np.mean(values[:h] ** 2))
            if not np.isfinite(mse) or mse <= 0:
                raise ValueError(f"{run['exp_id']}: log-scale MSE must be finite and positive.")
            records.append({
                "system_name": run["system_name"],
                "excitation_objective": run["excitation_objective"],
                "exp_id": run["exp_id"], "signal": s, "horizon": h, "mse": mse,
            })
    metrics = pd.DataFrame(records, columns=[
        "system_name", "excitation_objective", "exp_id", "signal", "horizon", "mse",
    ])

    fig, axes = plt.subplots(
        len(signals), 4, figsize=(figurewidth_in, 0.8* len(signals) + 1.0),
        squeeze=False, sharex=True, sharey=False, layout="constrained",
    )
    for col, (system, title) in enumerate(zip(systems, titles)):
        h = int(horizons.loc[system, "min"])
        axes[0, col].set_title(f"{title}\n" + r"$N_{\mathrm{steps}}" + f"={h}$", fontsize=9)
        system_metrics = metrics.loc[metrics["system_name"].eq(system)]

        for row, s in enumerate(signals):
            ax = axes[row, col]
            panel = system_metrics.loc[system_metrics["signal"].eq(s)]
            ax.set_xticks([])
            ax.set_xlim(-0.6, len(objective_labels) - 0.4)
            if col == 0:
                ax.set_ylabel(s.upper(), fontsize=9)
            if panel.empty:
                ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                        ha="center", va="center", color="0.45", fontsize=8)
                # Hide ticks locally; shared locators must stay intact.
                ax.tick_params(axis="both", which="both", bottom=False, top=False,
                               left=False, right=False, labelbottom=False, labelleft=False)
                continue

            for position, objective in enumerate(objective_labels):
                values = panel.loc[
                    panel["excitation_objective"].eq(objective), "mse",
                ].to_numpy()
                if values.size:
                    color = plt.get_cmap("tab10")(position)
                    if plot_style == "scatter":
                        offsets = rng.uniform(-jitter, jitter, size=values.size)
                        ax.scatter(
                            position + offsets, values, color=color,
                            s=6, alpha=0.75, edgecolors="none", zorder=3,
                        )
                    else:
                        ax.boxplot(
                            [values], positions=[position], widths=0.6,
                            patch_artist=True, manage_ticks=False,
                            boxprops={
                                "facecolor": color,
                                "edgecolor": color,
                                "alpha": 0.6,
                                "linewidth": 0.5,
                            },
                            medianprops={
                                "color": "black",
                                "linewidth": 0.5,
                            },
                            whiskerprops={"linewidth": 0.5},
                            capprops={"linewidth": 0.5},
                            flierprops={"marker": ".", "markeredgecolor": color},
                        )
            ax.tick_params(axis="y", labelsize=8)
            ax.grid(axis="y", which="major", alpha=0.3)
            ax.set_axisbelow(True)

            ax.set_yscale("log")
            lo = panel["mse"].min()
            hi = panel["mse"].max()
            padding = max(np.log10(hi / lo) * 0.06, 0.05)
            ax.set_ylim(lo / 10**padding, hi * 10**padding)
    
            ymin, ymax = ax.get_ylim()

            # Generate readable logarithmic tick candidates.
            ticks = mpl.ticker.LogLocator(
                base=10,
                subs=np.arange(1, 10, 0.5),
                numticks=100,
            ).tick_values(ymin, ymax)
             # Keep ticks at least 15% from each vertical edge.
            inset = 0.15 * np.log10(ymax / ymin)
            tick_min = ymin * 10**inset
            tick_max = ymax / 10**inset

            ticks = ticks[
                (ticks >= tick_min) & (ticks <= tick_max)
            ]

            # Keep at most three, spread across the logarithmic range.
            if ticks.size > 3:
                log_ticks = np.log10(ticks)
                targets = np.linspace(log_ticks[0], log_ticks[-1], 3)
                indices = np.abs(
                    log_ticks[:, None] - targets
                ).argmin(axis=0)
                ticks = ticks[np.unique(indices)]

            ax.yaxis.set_major_locator(mpl.ticker.FixedLocator(ticks))
            ax.yaxis.set_major_formatter(
                mpl.ticker.LogFormatterSciNotation(
                    labelOnlyBase=False,
                    minor_thresholds=(np.inf, np.inf),
                )
            )
            ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())

    legend_handles = []
    for position, (objective, label) in enumerate(objective_labels.items()):
        if not metrics["excitation_objective"].eq(objective).any():
            continue
        color = plt.get_cmap("tab10")(position)
        if plot_style == "scatter":
            handle = Line2D(
                [], [], linestyle="none", marker="o", markersize=5,
                markerfacecolor=color, markeredgecolor="none",
                alpha=0.75, label=label,
            )
        else:
            handle = Patch(facecolor=color, edgecolor=color, alpha=0.6, label=label)
        legend_handles.append(handle)
    fig.legend(
        handles=legend_handles,
        loc="outside lower center",
        title="Excitation objective",
        ncols=5, frameon=True, fontsize=8, title_fontsize=9,
        columnspacing=1.5, handletextpad=0.5,
    )

    fig.align_ylabels(axes[:, 0])
    fig.supylabel("Simulation MSE", fontsize=9)
    # fig.suptitle("Rollout performance at the training horizon", fontsize=9)
    return fig, axes, metrics


In [ ]:
outlier_id = "modeling-pendulum-gru-submission_nov-0309574b-seed42"

plot_data = modeling_results.loc[
    modeling_results["exp_id"].ne(outlier_id)
]

# plot_data = modeling_results


fig, axes, scores = plot_rollout_training_horizon(
    plot_data,
    plot_style="scatter",
    jitter=0.2,
    horizon=None,
)

if USE_CURV64:
    plt.savefig("paper_fig/submission_nov_curv64/quantitative_rollout_results_scatter.png", dpi=300, bbox_inches="tight")
    plt.savefig("paper_fig/submission_nov_curv64/quantitative_rollout_results_scatter.pdf", bbox_inches="tight")
else:
    plt.savefig("paper_fig/quantitative_rollout_results_scatter.png", dpi=300, bbox_inches="tight")
    plt.savefig("paper_fig/quantitative_rollout_results_scatter.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
outlier_id = "modeling-pendulum-gru-submission_nov-0309574b-seed42"

plot_data = modeling_results.loc[
    modeling_results["exp_id"].ne(outlier_id)
]

fig, axes, scores = plot_rollout_training_horizon(
    plot_data,
    plot_style="boxplot",
    jitter=0.2,
)

if USE_CURV64:
    plt.savefig("paper_fig/submission_nov_curv64/quantitative_rollout_results.png", dpi=300, bbox_inches="tight")
    plt.savefig("paper_fig/submission_nov_curv64/quantitative_rollout_results.pdf", bbox_inches="tight")
else:
    plt.savefig("paper_fig/quantitative_rollout_results.png", dpi=300, bbox_inches="tight")
    plt.savefig("paper_fig/quantitative_rollout_results.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)

# 1x4

In [ ]:
testing_columns = [
    f"rollout.{signal}.metrics.rmse_by_step"
    for signal in ("aprbs", "multisine", "ocp", "pi")
]


def mean_testing_rmse_by_step(run):
    traces = []
    for column in testing_columns:
        values = run[column]
        if values is None or (np.ndim(values) == 0 and pd.isna(values)):
            continue

        values = np.asarray(values, dtype=float)
        if values.ndim != 1 or values.size == 0:
            raise ValueError(
                f"{run['exp_id']}: invalid rollout data in {column}."
            )
        traces.append(values)

    if not traces:
        return None

    # Keep the steps available for every testing signal.
    # The existing plotting function then applies the chosen horizon.
    n_steps = min(map(len, traces))

    # The plotting function squares these values, giving the mean
    # signal MSE at each step.
    return np.sqrt(
        np.mean(
            np.stack([values[:n_steps] ** 2 for values in traces]),
            axis=0,
        )
    )


# Preserve the outlier exclusion used in the existing plot.
outlier_id = "modeling-pendulum-gru-submission_nov-0309574b-seed42"
plot_data_mean = modeling_results.loc[
    modeling_results["exp_id"].ne(outlier_id)
].copy()

plot_data_mean["rollout.mean_testing_signals.metrics.rmse_by_step"] = [
    mean_testing_rmse_by_step(run)
    for _, run in plot_data_mean.iterrows()
]

fig, axes, mean_scores = plot_rollout_training_horizon(
    plot_data_mean,
    signal="mean_testing_signals",
    plot_style="boxplot",  # Alternatively: "boxplot"
    jitter=0.2,
    horizon=None,
)

# Keep the shared "Rollout MSE" label and remove the signal row label.
axes[0, 0].set_ylabel("GRU")


from matplotlib.ticker import ScalarFormatter

for ax in axes.flat:
    formatter = ScalarFormatter(useMathText=True, useOffset=False)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))

    ax.yaxis.set_major_formatter(formatter)
    ax.yaxis.get_offset_text().set_fontsize(8)



# Optional export:
fig.savefig("paper_fig/quantitative_rollout_mean.png",
            dpi=300, bbox_inches="tight")
fig.savefig("paper_fig/quantitative_rollout_mean.pdf",
            bbox_inches="tight")

display(fig)
plt.close(fig)

## validation scores:

In [ ]:
OBJECTIVE_COLORS = {
    "dmpe_uniform": "#1f77b4",                  # blue
    "dmpe_static_dynamics": "#ff7f0e",          # orange
    "dmpe_static_mixture_dynamics": "#2ca02c",  # green
    "dmpe_static_curv": "#d62728",              # red
    "dmpe_static_mixture_curv": "#9467bd",      # purple
    "dmpe_static_LOLA_ls": "#8c564b",           # brown
}

In [ ]:
import json

METRIC = "rollout_training_horizon_mse"

system_titles = {
    #"fluid_tank": "Fluid tank",
    #"pendulum": "Pendulum",
    "cart_pole": "Cart-pole",
    "pmsm_nominal": "PMSM",
}
objective_labels = {
    "dmpe_uniform": "Uniform",
    "dmpe_static_dynamics": "Dynamics",
    # "dmpe_static_mixture_dynamics": "Mixture dynamics",
    "dmpe_static_curv": "Curvature",
    # "dmpe_static_mixture_curv": "Mixture curvature",
    #"dmpe_static_LOLA_ls": "LOLA",
}

# Read all completed runs, including their full validation histories.
runs = list_modeling_results(
    system_name=list(system_titles),
    model_type="gru",
    exp_name="submission_nov",
    status="completed",
)

frames = []
for run in runs:
    metadata = json.loads(
        (pathlib.Path(run["path"]) / "experiment.json").read_text()
    )
    frame = pd.DataFrame(metadata["validation"]["history"])[
        ["optimizer_steps", METRIC]
    ].rename(columns={METRIC: "score"})
    frame["system_name"] = metadata["system_name"]
    frame["objective"] = metadata["excitation_objective"]
    frame["exp_id"] = metadata["exp_id"]
    frames.append(frame)

if not frames:
    raise ValueError("No completed GRU submission_nov histories found.")

all_histories = pd.concat(frames, ignore_index=True)
if not np.isfinite(all_histories["score"]).all():
    raise ValueError("Some validation scores are non-finite; inspect all_histories.")

# Aggregate across runs at each recorded optimizer step.
summary = (
    all_histories.groupby(
        ["system_name", "objective", "optimizer_steps"]
    )["score"]
    .agg(mean="mean", variance=lambda x: x.var(ddof=0), n="count")
    .reset_index()
)

fig_curves, axes = plt.subplots(
    1, len(system_titles),
    figsize=(figurewidth_in, 2.5),
    squeeze=False,
    layout="constrained",
)

handles = {}
for ax, (system, title) in zip(axes.flat, system_titles.items()):
    panel = summary.loc[summary["system_name"].eq(system)]
    if panel.empty:
        ax.set_title(title)
        ax.text(
            0.5, 0.5, "No data",
            ha="center", va="center", transform=ax.transAxes,
        )
        continue

    floor = panel.loc[panel["mean"].gt(0), "mean"].min() * 0.5

    for i, (objective, label) in enumerate(objective_labels.items()):
        group = panel.loc[
            panel["objective"].eq(objective)
        ].sort_values("optimizer_steps")
        if group.empty:
            continue

        x = group["optimizer_steps"].to_numpy()
        mean = group["mean"].to_numpy()
        spread = jnp.sqrt(group["variance"].to_numpy())
        color = OBJECTIVE_COLORS[objective]

        line, = ax.plot(
            x, mean, color=color, linewidth=1.2, label=label,
        )
        ax.fill_between(
            x,
            np.maximum(mean - spread, floor),
            mean + spread,
            color=color, alpha=0.18, linewidth=0,
        )
        handles.setdefault(objective, line)

    ax.set(title=title, xlabel="Optimizer steps", yscale="log")
    ax.set_ylim(bottom=floor, top=0.1)
    ax.set_xlim(0, panel["optimizer_steps"].max())
    ax.grid(alpha=0.3)

fig_curves.supylabel("Validation rollout MSE")
fig_curves.suptitle(r"GRU: mean $\pm$ variance")
fig_curves.legend(
    handles=list(handles.values()),
    loc="outside lower center",
    ncols=3,
    frameon=False,
    fontsize=8,
)


plt.savefig("paper_fig/model_convergence.png", dpi=300, bbox_inches="tight")
plt.savefig("paper_fig/model_convergence.pdf", bbox_inches="tight")
display(fig_curves)
plt.close(fig_curves)

# Show how many runs contribute to each objective.
display(
    all_histories.groupby(["system_name", "objective"])["exp_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=list(system_titles), columns=list(objective_labels), fill_value=0)
    .rename(columns=objective_labels)
)